# Online Retail Order Data Platform — Phase 1
### Raw-to-Staging Ingestion Notebook (Object-Oriented Version)
**Name:** Llerin, Anton Uriel Medilo  
**Track:** Data Engineer — Midterm Activity 1  
**Date:** September 10, 2026



In [1]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess

# Auto-detect the project folder anywhere inside the mounted Drive
result = subprocess.run(
    ["find", "/content/drive", "-iname", "data_engineering_project", "-type", "d"],
    capture_output=True, text=True
)
matches = [line for line in result.stdout.splitlines() if line.strip()]

if not matches:
    raise FileNotFoundError(
        "Could not find a folder named 'data_engineering_project' anywhere in this Drive. "
        "Make sure it's shared with this account and a shortcut/copy exists somewhere in My Drive."
    )

PROJECT_PATH = matches[0]
print(f"Found project at: {PROJECT_PATH}")

Mounted at /content/drive
Found project at: /content/drive/MyDrive/data_engineering_project


## The `DataSource` class
One object per source system. Each instance knows its own file path, primary key, and expected schema, and knows how to load, verify, validate, and stage itself.



In [2]:
import pandas as pd


class DataSource:

    def __init__(self, name, raw_path, primary_key, expected_schema):
        self.name = name
        self.raw_path = raw_path
        self.primary_key = primary_key
        self.expected_schema = expected_schema
        self.df = None
        self.expected_records = None

    # ---------- Task 7: Ingestion ----------
    def load(self):
        """Load the raw CSV into a DataFrame. Returns self for easy chaining."""
        self.df = pd.read_csv(self.raw_path)
        return self

    # ---------- Task 8: Verify Ingestion ----------
    def verify(self, n_preview=3):
        print(f"===== {self.name.upper()} =====")
        print("Shape:", self.df.shape)
        display(self.df.head(n_preview))
        self.df.info()
        print()

    def ingestion_summary_row(self):
        return {
            "source": self.name,
            "expected_records": self.expected_records,
            "loaded_records": len(self.df),
            "columns": self.df.shape[1],
            "status": "Loaded" if len(self.df) > 0 else "FAILED",
        }

    # ---------- Task 9: Basic Schema Validation ----------
    def validate_schema(self):
        rows = []
        missing_cols = [c for c in self.expected_schema if c not in self.df.columns]
        for col, exp_type in self.expected_schema.items():
            if col not in self.df.columns:
                actual_type, observation = "MISSING", "Column missing from imported data"
            else:
                actual_type = str(self.df[col].dtype)
                observation = "OK" if actual_type == exp_type else f"Type mismatch (expected {exp_type})"
            rows.append({"field": col, "expected_type": exp_type, "actual_type": actual_type, "observation": observation})

        pk_present = self.primary_key in self.df.columns
        pk_nulls = self.df[self.primary_key].isna().sum() if pk_present else "PK missing"
        pk_dupes = self.df[self.primary_key].duplicated().sum() if pk_present else "PK missing"

        print(f"--- {self.name} ---")
        print(f"Primary key column '{self.primary_key}' present: {pk_present}")
        print(f"Primary key null count: {pk_nulls}")
        print(f"Primary key duplicate count: {pk_dupes}")
        print(f"Missing required columns: {missing_cols if missing_cols else 'None'}")
        validation_table = pd.DataFrame(rows)
        display(validation_table)
        print()
        return validation_table

    # ---------- Referential integrity (foreign key check against a parent DataSource) ----------
    def check_foreign_key(self, fk_column, parent):
        """parent is another DataSource whose primary_key this fk_column should reference."""
        orphans = self.df[~self.df[fk_column].isin(parent.df[parent.primary_key])]
        print(f"{self.name}.{fk_column} values not found in {parent.name}.{parent.primary_key}: {len(orphans)}")
        if len(orphans):
            display(orphans)
        return orphans

    # ---------- Task 10: Staging ----------
    def save_staging(self, staging_dir, prefix="stg_"):
        filename = f"{prefix}{self.name}.csv"
        out_path = f"{staging_dir}/{filename}"
        self.df.to_csv(out_path, index=False)
        print(f"Saved {out_path}  ({len(self.df)} rows)")
        return out_path


## Task 7 — Load the Five Raw Sources
One `DataSource` instance is created per file, each with its own primary key and expected schema, then loaded independently (not combined).

In [3]:
sources_config = {
    "customers": dict(
        raw_path=f"{PROJECT_PATH}/raw/customers.csv",
        primary_key="customer_id",
        expected_schema={
            "customer_id": "object", "full_name": "object", "email": "object",
            "city": "object", "registration_date": "object", "customer_type": "object"
        },
        expected_records=15,
    ),
    "products": dict(
        raw_path=f"{PROJECT_PATH}/raw/products.csv",
        primary_key="product_id",
        expected_schema={
            "product_id": "object", "product_name": "object", "category": "object",
            "unit_price": "float64", "supplier_id": "object", "stock_quantity": "int64"
        },
        expected_records=15,
    ),
    "orders": dict(
        raw_path=f"{PROJECT_PATH}/raw/orders.csv",
        primary_key="order_id",
        expected_schema={
            "order_id": "object", "customer_id": "object", "order_date": "object",
            "order_status": "object", "shipping_address": "object", "total_order_value": "float64"
        },
        expected_records=15,
    ),
    "payments": dict(
        raw_path=f"{PROJECT_PATH}/raw/payments.csv",
        primary_key="payment_id",
        expected_schema={
            "payment_id": "object", "order_id": "object", "payment_method": "object",
            "amount_paid": "float64", "payment_date": "object", "payment_status": "object"
        },
        expected_records=15,
    ),
    "deliveries": dict(
        raw_path=f"{PROJECT_PATH}/raw/deliveries.csv",
        primary_key="delivery_id",
        expected_schema={
            "delivery_id": "object", "order_id": "object", "courier": "object",
            "shipment_date": "object", "delivery_date": "object",
            "delivery_status": "object", "tracking_number": "object"
        },
        expected_records=15,
    ),
}

sources = {}
for name, cfg in sources_config.items():
    ds = DataSource(name, cfg["raw_path"], cfg["primary_key"], cfg["expected_schema"])
    ds.expected_records = cfg["expected_records"]
    ds.load()
    sources[name] = ds

print("All five raw sources loaded as DataSource objects.")

All five raw sources loaded as DataSource objects.


## Task 8 — Verify Ingestion
Each object verifies itself via its own `.verify()` method (wrapping `.shape`, `.head()`, `.info()`).

In [4]:
for ds in sources.values():
    ds.verify()

===== CUSTOMERS =====
Shape: (15, 6)


,customer_id,full_name,email,city,registration_date,customer_type
0,CUST001,Maria Santos,maria.santos@email.com,Iloilo City,2024-06-17,Premium
1,CUST002,Juan Reyes,juan.reyes@email.com,Makati,2025-03-04,Regular
2,CUST003,Ana Cruz,ana.cruz@email.com,Cagayan de Oro,2024-08-09,Premium


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   customer_id        15 non-null     object
 1   full_name          15 non-null     object
 2   email              15 non-null     object
 3   city               15 non-null     object
 4   registration_date  15 non-null     object
 5   customer_type      15 non-null     object
dtypes: object(6)
memory usage: 852.0+ bytes

===== PRODUCTS =====
Shape: (15, 6)


,product_id,product_name,category,unit_price,supplier_id,stock_quantity
0,P001,Wireless Mouse,Electronics,505.08,SUP01,250
1,P002,Bluetooth Speaker,Electronics,2184.83,SUP02,48
2,P003,USB-C Charging Cable,Electronics,1983.61,SUP02,46


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      15 non-null     object 
 1   product_name    15 non-null     object 
 2   category        15 non-null     object 
 3   unit_price      15 non-null     float64
 4   supplier_id     15 non-null     object 
 5   stock_quantity  15 non-null     int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 852.0+ bytes

===== ORDERS =====
Shape: (15, 6)


...

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           15 non-null     object 
 1   customer_id        15 non-null     object 
 2   order_date         15 non-null     object 
 3   order_status       15 non-null     object 
 4   shipping_address   15 non-null     object 
 5   total_order_value  15 non-null     float64
dtypes: float64(1), object(5)
memory usage: 852.0+ bytes

===== PAYMENTS =====
Shape: (15, 6)


,payment_id,order_id,payment_method,amount_paid,payment_date,payment_status
0,PAY2001,ORD1001,Credit Card,80.65,2025-08-17,Failed
1,PAY2002,ORD1002,Credit Card,2436.79,2025-06-24,Successful
2,PAY2003,ORD1003,GCash,5762.20,2025-08-09,Successful


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   payment_id      15 non-null     object 
 1   order_id        15 non-null     object 
 2   payment_method  15 non-null     object 
 3   amount_paid     15 non-null     float64
 4   payment_date    15 non-null     object 
 5   payment_status  15 non-null     object 
dtypes: float64(1), object(5)
memory usage: 852.0+ bytes

===== DELIVERIES =====
Shape: (15, 7)


,delivery_id,order_id,courier,shipment_date,delivery_date,delivery_status,tracking_number
0,DEL3001,ORD1001,LBC,NaN,NaN,Returned,TRK356598
1,DEL3002,ORD1002,Ninja Van,2025-06-25,2025-06-27,Delivered,TRK861416
2,DEL3003,ORD1003,GoGo Xpress,2025-08-08,2025-08-13,Delivered,TRK613864


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   delivery_id      15 non-null     object
 1   order_id         15 non-null     object
 2   courier          15 non-null     object
 3   shipment_date    12 non-null     object
 4   delivery_date    7 non-null      object
 5   delivery_status  15 non-null     object
 6   tracking_number  15 non-null     object
dtypes: object(7)
memory usage: 972.0+ bytes



### Ingestion Summary

| Source | Expected Records | Loaded Records | Columns | Status |
|---|---|---|---|---|
| customers | 15 | 15 | 6 | Loaded successfully |
| products | 15 | 15 | 6 | Loaded successfully |
| orders | 15 | 15 | 6 | Loaded successfully |
| payments | 15 | 15 | 6 | Loaded successfully — every payment maps to a real, existing order |
| deliveries | 15 | 15 | 7 | Loaded successfully — every order has exactly one delivery record |

Generated programmatically below via each object's `.ingestion_summary_row()` method. Per Task 5, the data was not intentionally made dirty — all foreign keys resolve cleanly.

In [5]:
ingestion_summary = pd.DataFrame([ds.ingestion_summary_row() for ds in sources.values()])
ingestion_summary

,source,expected_records,loaded_records,columns,status
0,customers,15,15,6,Loaded
1,products,15,15,6,Loaded
2,orders,15,15,6,Loaded
3,payments,15,15,6,Loaded
4,deliveries,15,15,7,Loaded


## Task 9 — Basic Schema Validation
Each object validates its own structure via `.validate_schema()`: required columns present, primary key non-null/unique, and Pandas dtypes compared against the expected schema.

In [6]:
validation_tables = {}
for name, ds in sources.items():
    validation_tables[name] = ds.validate_schema()

--- customers ---
Primary key column 'customer_id' present: True
Primary key null count: 0
Primary key duplicate count: 0
Missing required columns: None


,field,expected_type,actual_type,observation
0,customer_id,object,object,OK
1,full_name,object,object,OK
2,email,object,object,OK
3,city,object,object,OK
4,registration_date,object,object,OK
5,customer_type,object,object,OK



--- products ---
Primary key column 'product_id' present: True
Primary key null count: 0
Primary key duplicate count: 0
Missing required columns: None


,field,expected_type,actual_type,observation
0,product_id,object,object,OK
1,product_name,object,object,OK
2,category,object,object,OK
3,unit_price,float64,float64,OK
4,supplier_id,object,object,OK
5,stock_quantity,int64,int64,OK



--- orders ---
Primary key column 'order_id' present: True
Primary key null count: 0
Primary key duplicate count: 0
Missing required columns: None


,field,expected_type,actual_type,observation
0,order_id,object,object,OK
1,customer_id,object,object,OK
2,order_date,object,object,OK
3,order_status,object,object,OK
4,shipping_address,object,object,OK
5,total_order_value,float64,float64,OK



--- payments ---
Primary key column 'payment_id' present: True
Primary key null count: 0
Primary key duplicate count: 0
Missing required columns: None


,field,expected_type,actual_type,observation
0,payment_id,object,object,OK
1,order_id,object,object,OK
2,payment_method,object,object,OK
3,amount_paid,float64,float64,OK
4,payment_date,object,object,OK
5,payment_status,object,object,OK



--- deliveries ---
Primary key column 'delivery_id' present: True
Primary key null count: 0
Primary key duplicate count: 0
Missing required columns: None


,field,expected_type,actual_type,observation
0,delivery_id,object,object,OK
1,order_id,object,object,OK
2,courier,object,object,OK
3,shipment_date,object,object,OK
4,delivery_date,object,object,OK
5,delivery_status,object,object,OK
6,tracking_number,object,object,OK


### Referential Integrity Spot-Check
Not part of full cleaning yet, but worth flagging early: does every foreign key resolve back to a valid parent record? This uses the `check_foreign_key()` method, which takes the *parent* `DataSource` object as an argument.

In [7]:
orphan_orders_customers = sources["orders"].check_foreign_key("customer_id", sources["customers"])
orphan_payments_orders = sources["payments"].check_foreign_key("order_id", sources["orders"])
orphan_deliveries_orders = sources["deliveries"].check_foreign_key("order_id", sources["orders"])

orders.customer_id values not found in customers.customer_id: 0
payments.order_id values not found in orders.order_id: 0
deliveries.order_id values not found in orders.order_id: 0


**Finding:** all foreign keys resolve cleanly — every `orders.customer_id` exists in `customers`, every `payments.order_id` exists in `orders`, and every `deliveries.order_id` exists in `orders`. No orphan records were found in this raw dataset, consistent with Task 5's instruction that the data does not need to be intentionally made dirty at this stage. The `check_foreign_key()` method above is reusable — it can be called again on any future or larger dataset to catch referential-integrity problems before they reach a report. Task 4's guide question — what would happen if a foreign key referenced a value that doesn't exist in its parent source — is answered conceptually in the written report, since it doesn't occur in this particular dataset.

## Task 10 — Create the Staging Layer
Each object saves its own staging copy via `.save_staging()`, using the `stg_` naming convention. No transformation is applied — staging mirrors raw at this stage.

In [8]:
for ds in sources.values():
    ds.save_staging(staging_dir=f"{PROJECT_PATH}/staging")

print("\nPhase 1 flow complete: OPERATIONAL SOURCES -> RAW -> INGESTION -> STAGING")

Saved /content/drive/MyDrive/data_engineering_project/staging/stg_customers.csv  (15 rows)
Saved /content/drive/MyDrive/data_engineering_project/staging/stg_products.csv  (15 rows)
Saved /content/drive/MyDrive/data_engineering_project/staging/stg_orders.csv  (15 rows)
Saved /content/drive/MyDrive/data_engineering_project/staging/stg_payments.csv  (15 rows)
Saved /content/drive/MyDrive/data_engineering_project/staging/stg_deliveries.csv  (15 rows)

Phase 1 flow complete: OPERATIONAL SOURCES -> RAW -> INGESTION -> STAGING


## Phase 1 Completion Check
- All five raw sources were loaded independently (no joining/merging performed), each wrapped in its own `DataSource` object.
- Ingestion was verified using each object's `.verify()` method (shape, head, info).
- Basic schema validation confirmed required columns, primary key presence/uniqueness, and column data types via `.validate_schema()`, and a reusable `.check_foreign_key()` method confirmed referential integrity across all three FK relationships.
- Five staging copies (`stg_*.csv`) were written to `../staging/` via `.save_staging()`.
- **No integrated analytics dataset exists yet** — that begins in Phase 2.